# 🤖 Notebook 05 — LLM/VLM RCA Execution (Phase 2)

**Multimodal RCA Engine — Phase 2: Semantic Analysis & Generation**

Bu notebook, Multimodal RCA Motoru'nun kalbidir. Faz 1'de ayrıştırılan ve çıkarılan özellikleri 
alarak Büyük Dil Modelleri (LLMs) aracılığıyla semantik kök neden analizi (RCA) ve anomali tespiti yapar.

> ℹ️ **Desteklenen Ortamlar**: 
> - Local (Ollama) -> `qwen2.5:7b` veya `mistral:7b` gibi modeller (Yüksek Gizlilik)
> - Cloud (Google Gemini) -> `gemini-3-flash-preview` (Yüksek Performans / Hız)

---

## 5.1 — Kurulum ve Bağımlılıklar

Önce LLM konfigürasyonlarını (API key'leri vb.) yüklüyor ve `src/llm_engine.py` üzerinden motorumuzu başlatıyoruz.

In [ ]:
import sys
import os
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

PROJECT_ROOT = Path(os.getcwd()).parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import PARSED_DIR, DATA_DIR, RESULTS_DIR, print_section
from src.feature_extractor import LogFeatureExtractor
from src.llm_engine import LLMEngine
from src.rca_analyzer import SessionFormatter, PromptBuilder, ResultEvaluator

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("✅ Core dependencies loaded.")

In [ ]:
# Başlat: LLMEngine ve PromptBuilder
# LLMEngine .env üzerinden otomatik olarak 'ollama' veya 'gemini' backend'ini seçer

print_section("🚀 LLM Engine & Yapılandırma")

try:
    engine = LLMEngine()  # By default uses .env (Ollama or Gemini)
    prompt_builder = PromptBuilder()
    print(f"✅ LLM Engine aktif. Backend: {engine.backend.upper()}")
except Exception as e:
    print(f"❌ LLM Engine başlatılamadı: {e}")

## 5.2 — Veri Yükleme

Test edebilmek için Parsed HDFS verilerini ve özellik matrisi etiketlerini içeri alıyoruz.

In [ ]:
hdfs_parsed_path = PARSED_DIR / 'hdfs_parsed.csv'
labels_path = DATA_DIR / 'raw' / 'preprocessed' / 'anomaly_label.csv'


print("Veriler yükleniyor (biraz zaman alabilir)...")
if hdfs_parsed_path.exists():
    hdfs_df = pd.read_csv(hdfs_parsed_path)
    print(f"📊 HDFS logları yüklendi: {hdfs_df.shape}")
else:
    print("❌ HDFS parsed data bulunamadı.")

fe = LogFeatureExtractor()
labels_dict = fe.load_hdfs_labels(labels_path)

In [ ]:
# Analiz için bir anomali ve bir normal oturum blok kimliği seçelim
anomaly_blocks = [k for k, v in labels_dict.items() if v == 1]
normal_blocks = [k for k, v in labels_dict.items() if v == 0]

sample_anomaly = anomaly_blocks[0] if anomaly_blocks else None
sample_normal = normal_blocks[0] if normal_blocks else None

print(f"Test Anomali Block ID: {sample_anomaly}")
print(f"Test Normal Block ID: {sample_normal}")

## 5.3 — Tekli Oturum Analizi (DEMO)

Örnek olarak bir HDFS Session'ını hazırlayıp, modelden anomali durumunu değerlendirmesini isteyelim.

In [ ]:
def analyze_block(block_id, mode='classification'):
    """Helper to quickly analyze a given block_id with the current engine."""
    session_df = hdfs_df[hdfs_df['block_id'] == block_id]
    session_text = SessionFormatter.format_session_from_df(session_df, block_id, max_events=50)
    
    if mode == 'classification':
        sys_prompt, user_prompt = prompt_builder.build_anomaly_classification_prompt(session_text)
    else:
        sys_prompt, user_prompt = prompt_builder.build_root_cause_prompt(session_text, "")
        
    print_section(f"Analyzing {block_id} via {engine.backend.upper()}")
    response = engine.analyze(sys_prompt, user_prompt, temperature=0.1)
    
    return response

# 1. Test - Anomali Sınıflandırma (Normal Oturum Üzerinde)
if sample_normal:
    res_normal = analyze_block(sample_normal)
    print(json.dumps(res_normal, indent=2))

In [ ]:
# 2. Test - Anomali Sınıflandırma (Anormal Oturum Üzerinde)
if sample_anomaly:
    res_anomaly = analyze_block(sample_anomaly)
    print(json.dumps(res_anomaly, indent=2))

## 5.4 — Detaylı Root Cause Analysis (Kök Neden Analizi)

Anomali olduğu tespit edilen bir oturum için daha detaylı olan **Root Cause Analysis (RCA)** prompt'unu gönderelim.

In [ ]:
if sample_anomaly:
    res_rca = analyze_block(sample_anomaly, mode='rca')
    print_section("🔍 Kök Neden Analizi Çıktısı")
    print(json.dumps(res_rca, indent=2))
    
    # Visualizing output in a friendly way if it contains 'root_cause'
    if isinstance(res_rca, dict) and 'root_cause' in res_rca:
        print(f"\n🔥 Root Cause: {res_rca.get('root_cause')}")
        print(f"⚠️ Severity: {res_rca.get('severity')}")
        print(f"💡 Remediation:\n - " + "\n - ".join(res_rca.get('remediation', [])))

## 5.5 — Karşılaştırmalı Modelleme (Ollama vs Gemini)

`LLMEngine.set_backend()` fonksiyonu ile anlık olarak Ollama'dan Gemini API'ye geçiş yapabiliriz.

In [ ]:
try:
    # Backend'i Gemini'ye çevir
    engine.set_backend("gemini")
    
    # Aynı RCA isteğini Gemini'ye at
    if sample_anomaly:
        res_gemini = analyze_block(sample_anomaly, mode='rca')
        print(json.dumps(res_gemini, indent=2))
        
        # Tekrar Ollama'ya dön (isterseniz)
        engine.set_backend("ollama")
except Exception as e:
    print(f"Gemini test failed (Ensure GEMINI_API_KEY is correct). Error: {e}")

## 5.6 — Batch Evaluation (Küme Değerlendirmesi)

Algoritmanın genel performansını ölçmek için küçük bir örneklem (~50-100 oturum) üzerinde LLM çıkarımı yapalım ve Notebook 03'teki **Isolation Forest (F1=0.44)** sonucuyla kıyaslayalım.

> ℹ️ *Aşağıdaki hücre, LLM API limitlerine bağlı olarak biraz sürebilir. Test için 10'ar adetle sınırlandırılmıştır.*

In [ ]:
def evaluate_batch(engine, sample_size_normal=10, sample_size_anomaly=10):
    predictions = []
    ground_truth = []
    
    test_blocks = anomaly_blocks[:sample_size_anomaly] + normal_blocks[:sample_size_normal]
    
    print(f"Evaluating {len(test_blocks)} blocks using {engine.backend}...")
    for block_id in tqdm(test_blocks):
        session_df = hdfs_df[hdfs_df['block_id'] == block_id]
        session_text = SessionFormatter.format_session_from_df(session_df, block_id, max_events=30)
        sys_p, usr_p = prompt_builder.build_anomaly_classification_prompt(session_text)
        
        res = engine.analyze(sys_p, usr_p)
        
        # Ground truth 
        true_label = labels_dict[block_id]
        ground_truth.append(true_label)
        
        # Parse prediction
        pred_label = 0
        if isinstance(res, dict) and 'classification' in res:
            if res['classification'].upper() == "ANOMALY":
                pred_label = 1
        
        predictions.append(pred_label)
        
    return ground_truth, predictions


y_true, y_pred = evaluate_batch(engine, sample_size_normal=20, sample_size_anomaly=20)

metrics = ResultEvaluator.evaluate_classifications(y_true, y_pred)
ResultEvaluator.print_comparison_table(
     metrics_llm=metrics, 
     metrics_baseline={'precision': 0.48, 'recall': 0.41, 'f1': 0.44}, 
     llm_name=engine.backend.capitalize()
 )

In [ ]:
# === 🏆 TRIPLE MODEL COMPARISON: Llama vs Gemma vs Gemini ===
import time

all_results = {}

# --- 1. Llama 3.2 (1B) ---
print("=" * 60)
print("  🦙 TEST 1/3: Llama 3.2 (1B)")
print("=" * 60)
engine.set_backend("ollama")
engine.client.model = "llama3.2:1b"
y_true_l, y_pred_l = evaluate_batch(engine, sample_size_normal=20, sample_size_anomaly=20)
all_results["Llama 3.2 (1B)"] = ResultEvaluator.evaluate_classifications(y_true_l, y_pred_l)

# --- 2. Gemma 2 (2B) ---
print("\n" + "=" * 60)
print("  💎 TEST 2/3: Gemma 2 (2B)")
print("=" * 60)
engine.client.model = "gemma2:2b"
y_true_gm, y_pred_gm = evaluate_batch(engine, sample_size_normal=20, sample_size_anomaly=20)
all_results["Gemma 2 (2B)"] = ResultEvaluator.evaluate_classifications(y_true_gm, y_pred_gm)

# --- 3. Gemini Flash ---
print("\n" + "=" * 60)
print("  🌐 TEST 3/3: Gemini 3 Flash (Cloud)")
print("=" * 60)
engine.set_backend("gemini")
y_true_g, y_pred_g = evaluate_batch(engine, sample_size_normal=20, sample_size_anomaly=20)
all_results["Gemini Flash"] = ResultEvaluator.evaluate_classifications(y_true_g, y_pred_g)

# Reset to Ollama
engine.set_backend("ollama")

# === FINAL TABLE ===
bl = {'precision': 0.48, 'recall': 0.41, 'f1': 0.44}
print(f"\n{'='*80}")
print(f"  📊 BÜYÜK KARŞILAŞTIRMA: Baseline vs 3 LLM Modeli")
print(f"{'='*80}")
print(f"{'Metrik':<20} | {'Baseline':<12} | {'Llama 1B':<12} | {'Gemma 2B':<12} | {'Gemini':<12}")
print("-" * 80)

for metric_name, key in [("Precision", "precision"), ("Recall", "recall"), ("F1 Score", "f1")]:
    row = f"{metric_name:<20} | {bl[key]:<12.2f} |"
    for model_name in ["Llama 3.2 (1B)", "Gemma 2 (2B)", "Gemini Flash"]:
        row += f" {all_results[model_name][key]:<12.2f} |"
    print(row)

print("-" * 80)

# Find the winner
best_name = "Baseline"
best_f1 = bl['f1']
for name, m in all_results.items():
    if m['f1'] > best_f1:
        best_name, best_f1 = name, m['f1']

print(f"\n🏆 EN İYİ MODEL: {best_name} (F1 = {best_f1:.2f})")


In [10]:
# === Gemini Batch Evaluation ===
print("🔄 Switching to Gemini...")
engine.set_backend("gemini")

y_true_g, y_pred_g = evaluate_batch(engine, sample_size_normal=20, sample_size_anomaly=20)
metrics_gemini = ResultEvaluator.evaluate_classifications(y_true_g, y_pred_g)

# Switch back to Ollama
engine.set_backend("ollama")

# === 3-WAY COMPARISON TABLE ===
print(f"\n{'='*70}")
print(f"  📊 KARŞILAŞTIRMA: Baseline vs Ollama vs Gemini")
print(f"{'='*70}")
print(f"{'Metrik':<22} | {'Baseline (IF)':<15} | {'Ollama':<15} | {'Gemini':<15}")
print("-" * 70)

bl = {'precision': 0.48, 'recall': 0.41, 'f1': 0.44}
print(f"{'Anomali Precision':<22} | {bl['precision']:<15.2f} | {metrics['precision']:<15.2f} | {metrics_gemini['precision']:<15.2f}")
print(f"{'Anomali Recall':<22} | {bl['recall']:<15.2f} | {metrics['recall']:<15.2f} | {metrics_gemini['recall']:<15.2f}")
print(f"{'F1 Score':<22} | {bl['f1']:<15.2f} | {metrics['f1']:<15.2f} | {metrics_gemini['f1']:<15.2f}")
print("-" * 70)

# Find the winner
best_name = "Baseline"
best_f1 = bl['f1']
if metrics['f1'] > best_f1:
    best_name, best_f1 = "Ollama (llama3.2:1b)", metrics['f1']
if metrics_gemini['f1'] > best_f1:
    best_name, best_f1 = "Gemini (flash)", metrics_gemini['f1']

print(f"\n🏆 EN İYİ MODEL: {best_name} (F1 = {best_f1:.2f})")


KeyboardInterrupt: 